In [7]:
from datetime import datetime
print("Run timestamp (UTC):", datetime.utcnow().isoformat())

try:
    from pyspark.sql import SparkSession
    import pyspark, sys, platform, os
    spark = (
        SparkSession.builder
        .appName("BDA-Lab0")
        .config("spark.sql.session.timeZone","UTC")
        .config("spark.sql.shuffle.partitions","8")
        .getOrCreate()
    )
    print("Spark:", spark.version)
    print("PySpark:", pyspark.__version__)
    print("Python:", sys.version.split()[0], "|", platform.platform())
    print("SPARK_HOME:", os.environ.get("SPARK_HOME", "<pip-only>"))
except Exception as e:
    print("Spark init failed:", e)
    spark = None

Run timestamp (UTC): 2025-10-23T07:47:47.600114
Spark: 4.0.1
PySpark: 4.0.1
Python: 3.10.19 | Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
SPARK_HOME: <pip-only>


In [8]:
if spark is None:
    raise SystemExit("Spark not available. Fix setup and re-run Section 1.")

data = [("a",1),("b",2),("c",3),("a",2)]
df = spark.createDataFrame(data, ["key","val"])
df.show()
df.groupBy("key").count().show()

print("\n--- formatted plan ---")
df.groupBy("key").count().explain(mode="formatted")

+---+---+
|key|val|
+---+---+
|  a|  1|
|  b|  2|
|  c|  3|
|  a|  2|
+---+---+



[Stage 10:=====================================================>  (23 + 1) / 24]

+---+-----+
|key|count|
+---+-----+
|  a|    2|
|  b|    1|
|  c|    1|
+---+-----+


--- formatted plan ---
== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [2]: [key#38, val#39L]
Arguments: [key#38, val#39L], MapPartitionsRDD[18] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [1]: [key#38]
Input [2]: [key#38, val#39L]

(3) HashAggregate
Input [1]: [key#38]
Keys [1]: [key#38]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#64L]
Results [2]: [key#38, count#65L]

(4) Exchange
Input [2]: [key#38, count#65L]
Arguments: hashpartitioning(key#38, 8), ENSURE_REQUIREMENTS, [plan_id=161]

(5) HashAggregate
Input [2]: [key#38, count#65L]
Keys [1]: [key#38]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#63L]
Results [2]: [key#38, count(1)#63L AS 

In [9]:
rdd = spark.sparkContext.parallelize([1,2,3,4,5])
print(rdd.map(lambda x: x*2).collect())

[2, 4, 6, 8, 10]


In [10]:
# Cellule 4 : Save evidence (corrigée)
from io import StringIO
from pathlib import Path
from pyspark.sql.functions import col # <-- ASSUREZ-VOUS QUE CECI EST IMPORTÉ

if spark is None:
    print("Impossible de sauvegarder le plan : Spark non disponible.")
    raise SystemExit(1)

buf = StringIO()
old_stdout = sys.stdout
try:
    sys.stdout = buf
    # Ligne corrigée : Utilise .groupBy(col("id") % 2) au lieu de .groupByExpr("id % 2")
    spark.range(10).groupBy(col("id") % 2).count().explain(mode="formatted") 
finally:
    sys.stdout = old_stdout

file_path = Path("lab0_plan.txt")
file_path.write_text(buf.getvalue(), encoding="utf-8")

print(f"Plan d'exécution sauvegardé dans : {file_path.resolve()}")

Plan d'exécution sauvegardé dans : /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab0_plan.txt
